<a href="https://colab.research.google.com/github/Uchihatt/My-Projects/blob/main/2_model_approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install scikit-learn
import numpy as np
import pandas as pd

np.random.seed(42)

N_CUSTOMERS = 6000
DAYS = 120
START = pd.Timestamp("2025-01-01")

# Customer table
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(5)}" for i in range(N_CUSTOMERS)],
    "tenure_days": np.random.randint(10, 1200, N_CUSTOMERS),
    "region": np.random.choice(["Lahore","Karachi","Islamabad","Faisalabad","Multan"], N_CUSTOMERS, p=[.30,.30,.15,.15,.10]),
    "acq_channel": np.random.choice(["Paid Social","Paid Search","Organic","Referral","CRM"], N_CUSTOMERS, p=[.25,.20,.25,.15,.15]),
    "past_30d_orders": np.random.poisson(1.2, N_CUSTOMERS),
    "past_30d_spend": np.random.gamma(2.0, 2500, N_CUSTOMERS),   # skewed
    "discount_affinity": np.clip(np.random.normal(0.25, 0.15, N_CUSTOMERS), 0, 1),  # 0-1
})

# Latent: price sensitivity + baseline propensity
customers["price_sensitivity"] = np.clip(0.6*customers["discount_affinity"] + np.random.normal(0,0.15,N_CUSTOMERS), 0, 1)
customers["baseline_propensity"] = np.clip(
    0.25 + 0.08*np.log1p(customers["past_30d_spend"]/1000) + 0.10*customers["past_30d_orders"] - 0.12*customers["price_sensitivity"]
    + np.random.normal(0,0.05,N_CUSTOMERS),
    0.01, 0.95
)

# Promo campaign window
CAMPAIGN_START = START + pd.Timedelta(days=45)
CAMPAIGN_END   = START + pd.Timedelta(days=75)

# Targeting rule (biased): more likely to target discount-affine + active users
logit = (
    -1.0
    + 1.6*customers["price_sensitivity"]
    + 0.5*np.log1p(customers["past_30d_orders"])
    + 0.35*np.log1p(customers["past_30d_spend"]/1000)
    + np.where(customers["acq_channel"].eq("CRM"), 0.4, 0.0)
)
p_treat = 1/(1+np.exp(-logit))
customers["treated"] = (np.random.rand(N_CUSTOMERS) < p_treat).astype(int)

# Generate daily purchase events (conversion) and orders
rows = []
order_id = 0

categories = ["Grocery","Fashion","Electronics","Beauty","Home","Pharmacy"]
cat_mult = {"Grocery":0.8,"Fashion":1.1,"Electronics":2.0,"Beauty":1.0,"Home":1.4,"Pharmacy":0.9}

for day in range(DAYS):
    date = START + pd.Timedelta(days=day)
    in_campaign = (date >= CAMPAIGN_START) & (date <= CAMPAIGN_END)

    # Conversion probability base (per day)
    base = customers["baseline_propensity"]/30.0  # monthly->daily

    # Treatment effect:
    # - During campaign, treated customers have higher conversion probability (uplift)
    # - But part of uplift is pull-forward: conversion shifts earlier, not purely incremental
    uplift = np.zeros(N_CUSTOMERS)

    if in_campaign:
        # true uplift depends on price sensitivity (deal seekers respond more)
        uplift = 0.020 + 0.040*customers["price_sensitivity"]   # +2% to +6% absolute daily increment (small)
        uplift *= customers["treated"]

    # Pull-forward cannibalization: treated customers who convert during campaign are less likely to convert after
    # We'll implement via "post-campaign suppression" later using a rolling flag.
    p_buy = np.clip(base + uplift, 0, 0.35)

    buy = (np.random.rand(N_CUSTOMERS) < p_buy)

    buyers = customers.loc[buy, ["customer_id","treated","price_sensitivity"]].copy()
    if len(buyers) == 0:
        continue

    for _, b in buyers.iterrows():
        order_id += 1
        cat = np.random.choice(categories, p=[.20,.18,.12,.15,.20,.15])
        gross = np.random.gamma(2.0, 1800) * cat_mult[cat] + 200

        # Discount only if treated and in campaign (promo redemption behavior)
        redeemed = int((b["treated"] == 1) and in_campaign and (np.random.rand() < (0.35 + 0.45*b["price_sensitivity"])))
        disc_pct = 0.0
        if redeemed:
            disc_pct = np.clip(np.random.normal(0.18 + 0.10*b["price_sensitivity"], 0.06), 0.05, 0.55)

        net = gross*(1-disc_pct)

        # COGS and margin
        cogs = gross*np.random.uniform(0.55, 0.75)
        margin = net - cogs

        rows.append({
            "order_id": f"O{order_id:08d}",
            "customer_id": b["customer_id"],
            "order_date": date,
            "category": cat,
            "gross_revenue": round(gross,2),
            "discount_pct": round(disc_pct,3),
            "discount_amount": round(gross*disc_pct,2),
            "net_revenue": round(net,2),
            "cogs": round(cogs,2),
            "gross_margin": round(margin,2),
            "treated": int(b["treated"]),
            "in_campaign": int(in_campaign),
            "redeemed": redeemed
        })

orders = pd.DataFrame(rows)

customers.to_csv("promo_customers.csv", index=False)
orders.to_csv("promo_orders.csv", index=False)

print("Saved promo_customers.csv", customers.shape)
print("Saved promo_orders.csv", orders.shape)
orders.head()


Saved promo_customers.csv (6000, 10)
Saved promo_orders.csv (14320, 13)


,order_id,customer_id,order_date,category,gross_revenue,discount_pct,discount_amount,net_revenue,cogs,gross_margin,treated,in_campaign,redeemed
0,O00000001,C00106,2025-01-01,Home,2620.21,0.0,0.0,2620.21,1831.93,788.28,1,0,0
1,O00000002,C00112,2025-01-01,Beauty,1079.92,0.0,0.0,1079.92,607.63,472.29,1,0,0
2,O00000003,C00140,2025-01-01,Home,7073.98,0.0,0.0,7073.98,4305.72,2768.26,1,0,0
3,O00000004,C00144,2025-01-01,Home,3839.29,0.0,0.0,3839.29,2336.04,1503.25,1,0,0
4,O00000005,C00209,2025-01-01,Electronics,7007.12,0.0,0.0,7007.12,4695.03,2312.09,0,0,0


In [2]:
import pandas as pd
import numpy as np

customers = pd.read_csv("promo_customers.csv")
orders = pd.read_csv("promo_orders.csv", parse_dates=["order_date"])

START = pd.Timestamp("2025-01-01")
CAMPAIGN_START = START + pd.Timedelta(days=45)
CAMPAIGN_END   = START + pd.Timedelta(days=75)

PRE_START  = START + pd.Timedelta(days=15)
PRE_END    = CAMPAIGN_START - pd.Timedelta(days=1)

POST_START = CAMPAIGN_END + pd.Timedelta(days=1)
POST_END   = START + pd.Timedelta(days=105)

def period(d):
    if PRE_START <= d <= PRE_END: return "pre"
    if CAMPAIGN_START <= d <= CAMPAIGN_END: return "during"
    if POST_START <= d <= POST_END: return "post"
    return None

orders["period"] = orders["order_date"].apply(period)
orders = orders.dropna(subset=["period"])

# Customer-period aggregates
cp = orders.groupby(["customer_id","period"]).agg(
    orders=("order_id","count"),
    net_revenue=("net_revenue","sum"),
    gross_margin=("gross_margin","sum"),
    discount_amount=("discount_amount","sum"),
    redeemed=("redeemed","sum")
).reset_index()

# pivot to wide
wide = cp.pivot(index="customer_id", columns="period")
wide.columns = [f"{a}_{b}" for a,b in wide.columns]
wide = wide.reset_index().merge(customers[["customer_id","treated","price_sensitivity","discount_affinity","past_30d_orders","past_30d_spend","tenure_days","region","acq_channel"]], on="customer_id", how="left")

# fill missing with 0
for c in wide.columns:
    if c.startswith(("orders_","net_revenue_","gross_margin_","discount_amount_","redeemed_")):
        wide[c] = wide[c].fillna(0)

# outcomes (conversion proxy: any order during campaign)
wide["converted_during"] = (wide["orders_during"] > 0).astype(int)
wide["converted_post"]   = (wide["orders_post"] > 0).astype(int)

wide.head()


,customer_id,orders_during,orders_post,orders_pre,net_revenue_during,net_revenue_post,net_revenue_pre,gross_margin_during,gross_margin_post,gross_margin_pre,...,treated,price_sensitivity,discount_affinity,past_30d_orders,past_30d_spend,tenure_days,region,acq_channel,converted_during,converted_post
0,C00000,0.0,0.0,1.0,0.00,0.00,10388.13,0.00,0.00,2930.35,...,1,0.265033,0.354350,1,4190.802558,1136,Faisalabad,Organic,0,0
1,C00001,1.0,0.0,1.0,827.06,0.00,1611.86,243.67,0.00,597.42,...,0,0.309179,0.253019,4,1383.175278,870,Lahore,CRM,1,0
2,C00003,0.0,0.0,1.0,0.00,0.00,11503.22,0.00,0.00,4522.97,...,1,0.000000,0.188887,0,3558.048668,1105,Multan,Paid Social,0,0
3,C00007,2.0,0.0,1.0,6952.97,0.00,7051.66,483.01,0.00,2116.79,...,1,0.329301,0.191414,1,3749.039040,340,Multan,Organic,1,0
4,C00008,1.0,1.0,1.0,5049.10,2334.07,4739.03,1887.12,940.89,2039.61,...,0,0.153887,0.117480,1,6019.682185,97,Faisalabad,Organic,1,1


In [3]:
import pandas as pd
import numpy as np

customers = pd.read_csv("promo_customers.csv")
orders = pd.read_csv("promo_orders.csv", parse_dates=["order_date"])

START = pd.Timestamp("2025-01-01")
CAMPAIGN_START = START + pd.Timedelta(days=45)
CAMPAIGN_END   = START + pd.Timedelta(days=75)

PRE_START  = START + pd.Timedelta(days=15)
PRE_END    = CAMPAIGN_START - pd.Timedelta(days=1)

POST_START = CAMPAIGN_END + pd.Timedelta(days=1)
POST_END   = START + pd.Timedelta(days=105)

def period(d):
    if PRE_START <= d <= PRE_END: return "pre"
    if CAMPAIGN_START <= d <= CAMPAIGN_END: return "during"
    if POST_START <= d <= POST_END: return "post"
    return None

orders["period"] = orders["order_date"].apply(period)
orders = orders.dropna(subset=["period"])

# Customer-period aggregates
cp = orders.groupby(["customer_id","period"]).agg(
    orders=("order_id","count"),
    net_revenue=("net_revenue","sum"),
    gross_margin=("gross_margin","sum"),
    discount_amount=("discount_amount","sum"),
    redeemed=("redeemed","sum")
).reset_index()

# pivot to wide
wide = cp.pivot(index="customer_id", columns="period")
wide.columns = [f"{a}_{b}" for a,b in wide.columns]
wide = wide.reset_index().merge(customers[["customer_id","treated","price_sensitivity","discount_affinity","past_30d_orders","past_30d_spend","tenure_days","region","acq_channel"]], on="customer_id", how="left")

# fill missing with 0
for c in wide.columns:
    if c.startswith(("orders_","net_revenue_","gross_margin_","discount_amount_","redeemed_")):
        wide[c] = wide[c].fillna(0)

# outcomes (conversion proxy: any order during campaign)
wide["converted_during"] = (wide["orders_during"] > 0).astype(int)
wide["converted_post"]   = (wide["orders_post"] > 0).astype(int)

wide.head()


,customer_id,orders_during,orders_post,orders_pre,net_revenue_during,net_revenue_post,net_revenue_pre,gross_margin_during,gross_margin_post,gross_margin_pre,...,treated,price_sensitivity,discount_affinity,past_30d_orders,past_30d_spend,tenure_days,region,acq_channel,converted_during,converted_post
0,C00000,0.0,0.0,1.0,0.00,0.00,10388.13,0.00,0.00,2930.35,...,1,0.265033,0.354350,1,4190.802558,1136,Faisalabad,Organic,0,0
1,C00001,1.0,0.0,1.0,827.06,0.00,1611.86,243.67,0.00,597.42,...,0,0.309179,0.253019,4,1383.175278,870,Lahore,CRM,1,0
2,C00003,0.0,0.0,1.0,0.00,0.00,11503.22,0.00,0.00,4522.97,...,1,0.000000,0.188887,0,3558.048668,1105,Multan,Paid Social,0,0
3,C00007,2.0,0.0,1.0,6952.97,0.00,7051.66,483.01,0.00,2116.79,...,1,0.329301,0.191414,1,3749.039040,340,Multan,Organic,1,0
4,C00008,1.0,1.0,1.0,5049.10,2334.07,4739.03,1887.12,940.89,2039.61,...,0,0.153887,0.117480,1,6019.682185,97,Faisalabad,Organic,1,1


In [4]:
summary_naive = wide.groupby("treated").agg(
    customers=("customer_id","count"),
    conv_rate=("converted_during","mean"),
    avg_net=("net_revenue_during","mean"),
    avg_margin=("gross_margin_during","mean"),
    avg_discount=("discount_amount_during","mean"),
)
summary_naive


,customers,conv_rate,avg_net,avg_margin,avg_discount
treated,,,,,
0,1939,0.492006,2717.778798,954.757581,0.000000
1,3007,0.820419,6062.241959,1758.365298,572.133898


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

psm_df = wide.copy()

# Use ONLY pre-treatment features (safe)
feature_cols_num = ["price_sensitivity","discount_affinity","past_30d_orders","past_30d_spend","tenure_days"]
feature_cols_cat = ["region","acq_channel"]

X = psm_df[feature_cols_num + feature_cols_cat]
y = psm_df["treated"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", feature_cols_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat)
    ]
)

psm_model = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=200))
])

psm_model.fit(X, y)
psm_df["propensity"] = psm_model.predict_proba(X)[:,1]
psm_df[["treated","propensity"]].head()


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,treated,propensity
0,1,0.610364
1,0,0.740268
2,1,0.448685
3,1,0.622693
4,0,0.583167


In [6]:
treated = psm_df[psm_df["treated"]==1].copy()
control = psm_df[psm_df["treated"]==0].copy()

treated = treated.sort_values("propensity")
control = control.sort_values("propensity")

# For each treated, match closest control (with replacement)
control_props = control["propensity"].to_numpy()
control_ids = control["customer_id"].to_numpy()

def match_one(p):
    idx = np.argmin(np.abs(control_props - p))
    return control_ids[idx]

treated["matched_control_id"] = treated["propensity"].apply(match_one)
matched = treated.merge(control.add_prefix("ctrl_"), left_on="matched_control_id", right_on="ctrl_customer_id", how="left")

matched.shape, matched.head()


((3007, 55),
   customer_id  orders_during  orders_post  orders_pre  net_revenue_during  \
 0      C01292            4.0          0.0         0.0            13763.16   
 1      C05870            1.0          0.0         0.0             8978.77   
 2      C00272            0.0          1.0         0.0                0.00   
 3      C01056            1.0          0.0         0.0             3153.16   
 4      C03820            1.0          1.0         1.0             2233.62   
 
    net_revenue_post  net_revenue_pre  gross_margin_during  gross_margin_post  \
 0              0.00             0.00              3800.16               0.00   
 1              0.00             0.00              2494.62               0.00   
 2           2012.32             0.00                 0.00             776.09   
 3              0.00             0.00               873.48               0.00   
 4           7207.72          3982.83               645.34            2885.11   
 
    gross_margin_pre  ...  ct

In [7]:
# Treated outcomes
t_conv = matched["converted_during"].mean()
t_net  = matched["net_revenue_during"].mean()
t_mar  = matched["gross_margin_during"].mean()

# Matched control outcomes
c_conv = matched["ctrl_converted_during"].mean()
c_net  = matched["ctrl_net_revenue_during"].mean()
c_mar  = matched["ctrl_gross_margin_during"].mean()

uplift = {
    "uplift_conv_rate": t_conv - c_conv,
    "uplift_net_revenue_per_customer": t_net - c_net,
    "uplift_margin_per_customer": t_mar - c_mar
}
uplift


{'uplift_conv_rate': np.float64(0.3185899567675424),
 'uplift_net_revenue_per_customer': np.float64(3203.997083471899),
 'uplift_margin_per_customer': np.float64(767.2415297638842)}

In [8]:
t_post = matched["converted_post"].mean()
c_post = matched["ctrl_converted_post"].mean()

cannibal = {
    "post_conv_treated": t_post,
    "post_conv_control": c_post,
    "post_delta": t_post - c_post  # negative suggests cannibalization
}
cannibal


{'post_conv_treated': np.float64(0.42733621549717327),
 'post_conv_control': np.float64(0.5184569338210842),
 'post_delta': np.float64(-0.09112071832391089)}

In [9]:
# incremental orders during vs incremental orders post
t_orders_during = matched["orders_during"].mean()
c_orders_during = matched["ctrl_orders_during"].mean()
t_orders_post   = matched["orders_post"].mean()
c_orders_post   = matched["ctrl_orders_post"].mean()

inc_during = t_orders_during - c_orders_during
inc_post   = t_orders_post - c_orders_post

pull_forward_index = np.nan
if inc_during > 0:
    pull_forward_index = max(0, -inc_post) / inc_during  # share of uplift "taken back" after campaign

{"inc_orders_during": inc_during, "inc_orders_post": inc_post, "pull_forward_index": pull_forward_index}


{'inc_orders_during': np.float64(0.8919188560026604),
 'inc_orders_post': np.float64(-0.12271366810774864),
 'pull_forward_index': np.float64(0.13758389261744974)}

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_features = feature_cols_num + feature_cols_cat

def build_clf():
    return Pipeline(steps=[
        ("prep", preprocess),
        ("clf", RandomForestClassifier(n_estimators=250, max_depth=10, random_state=42, n_jobs=-1))
    ])

train = psm_df.copy()
Xall = train[model_features]
yall = train["converted_during"].astype(int)

treated_mask = train["treated"]==1
control_mask = train["treated"]==0

m_t = build_clf()
m_c = build_clf()

m_t.fit(Xall[treated_mask], yall[treated_mask])
m_c.fit(Xall[control_mask], yall[control_mask])

p_t = m_t.predict_proba(Xall)[:,1]
p_c = m_c.predict_proba(Xall)[:,1]

train["uplift_score"] = p_t - p_c
train[["customer_id","treated","uplift_score"]].head()


,customer_id,treated,uplift_score
0,C00000,1,0.060771
1,C00001,0,0.120756
2,C00003,1,0.162213
3,C00007,1,0.459935
4,C00008,0,0.211944


In [11]:
train["uplift_decile"] = pd.qcut(train["uplift_score"], 10, labels=False, duplicates="drop")

uplift_by_decile = train.groupby("uplift_decile").agg(
    customers=("customer_id","count"),
    avg_uplift=("uplift_score","mean"),
    treated_rate=("treated","mean"),
    actual_conv=("converted_during","mean")
).sort_index(ascending=False)

uplift_by_decile


,customers,avg_uplift,treated_rate,actual_conv
uplift_decile,,,,
9,495,0.527110,0.436364,0.438384
8,494,0.450023,0.572874,0.574899
7,495,0.409075,0.666667,0.672727
6,494,0.380085,0.724696,0.732794
5,495,0.351547,0.721212,0.777778
4,494,0.319116,0.742915,0.852227
3,495,0.281945,0.676768,0.870707
2,494,0.237957,0.560729,0.819838
1,495,0.171905,0.432323,0.701010


In [12]:
treated_n = matched.shape[0]

scorecard = {
    "treated_customers_evaluated": treated_n,
    "incremental_conv_rate": uplift["uplift_conv_rate"],
    "incremental_net_rev_per_customer": uplift["uplift_net_revenue_per_customer"],
    "incremental_margin_per_customer": uplift["uplift_margin_per_customer"],
    "total_incremental_net_revenue": uplift["uplift_net_revenue_per_customer"] * treated_n,
    "total_incremental_margin": uplift["uplift_margin_per_customer"] * treated_n,
    "pull_forward_index": pull_forward_index,
    "cannibalization_flag": int(pull_forward_index is not None and pull_forward_index > 0.35)
}

pd.DataFrame([scorecard])


,treated_customers_evaluated,incremental_conv_rate,incremental_net_rev_per_customer,incremental_margin_per_customer,total_incremental_net_revenue,total_incremental_margin,pull_forward_index,cannibalization_flag
0,3007,0.31859,3203.997083,767.24153,9634419.23,2307095.28,0.137584,0


In [13]:
treated_n = matched.shape[0]

scorecard = {
    "treated_customers_evaluated": treated_n,
    "incremental_conv_rate": uplift["uplift_conv_rate"],
    "incremental_net_rev_per_customer": uplift["uplift_net_revenue_per_customer"],
    "incremental_margin_per_customer": uplift["uplift_margin_per_customer"],
    "total_incremental_net_revenue": uplift["uplift_net_revenue_per_customer"] * treated_n,
    "total_incremental_margin": uplift["uplift_margin_per_customer"] * treated_n,
    "pull_forward_index": pull_forward_index,
    "cannibalization_flag": int(pull_forward_index is not None and pull_forward_index > 0.35)
}

pd.DataFrame([scorecard])


,treated_customers_evaluated,incremental_conv_rate,incremental_net_rev_per_customer,incremental_margin_per_customer,total_incremental_net_revenue,total_incremental_margin,pull_forward_index,cannibalization_flag
0,3007,0.31859,3203.997083,767.24153,9634419.23,2307095.28,0.137584,0


In [14]:
pd.DataFrame([scorecard]).to_csv("promo_scorecard.csv", index=False)
print("Saved promo_scorecard.csv")


Saved promo_scorecard.csv


In [15]:
during = orders[orders["period"]=="during"].copy()

promo_feat = during.groupby("customer_id").agg(
    orders_during=("order_id","count"),
    redeemed_orders=("redeemed","sum"),
    avg_discount_pct=("discount_pct","mean"),
    pct_orders_discounted=("discount_pct", lambda x: (x > 0.10).mean()),
    avg_margin=("gross_margin","mean"),
    net_rev=("net_revenue","sum"),
).reset_index()

promo_feat = promo_feat.merge(customers[["customer_id","treated","discount_affinity","price_sensitivity"]], on="customer_id", how="left")
promo_feat = promo_feat.fillna(0)

promo_feat.head()


,customer_id,orders_during,redeemed_orders,avg_discount_pct,pct_orders_discounted,avg_margin,net_rev,treated,discount_affinity,price_sensitivity
0,C00001,1,0,0.00000,0.00,243.6700,827.06,0,0.253019,0.309179
1,C00007,2,2,0.15300,0.50,241.5050,6952.97,1,0.191414,0.329301
2,C00008,1,0,0.00000,0.00,1887.1200,5049.10,0,0.117480,0.153887
3,C00009,2,0,0.00000,0.00,1151.1400,7280.69,0,0.309342,0.242616
4,C00011,4,1,0.04475,0.25,1146.1975,13933.56,1,0.301067,0.188912


In [16]:
def flag_discount_hunter(row):
    # Tune thresholds based on your data
    cond1 = row["pct_orders_discounted"] >= 0.75
    cond2 = row["avg_discount_pct"] >= 0.22
    cond3 = row["redeemed_orders"] >= 2
    cond4 = row["avg_margin"] <= np.percentile(promo_feat["avg_margin"], 20)
    return int((cond1 and cond2) or (cond2 and cond3 and cond4))

promo_feat["discount_hunter_flag"] = promo_feat.apply(flag_discount_hunter, axis=1)

promo_feat["discount_hunter_flag"].value_counts()


,count
discount_hunter_flag,
0,3208
1,213


In [17]:
promo_feat.sort_values(["discount_hunter_flag","avg_discount_pct","pct_orders_discounted"], ascending=False)\
          .to_csv("discount_hunter_flags.csv", index=False)
print("Saved discount_hunter_flags.csv")


Saved discount_hunter_flags.csv
